In [ ]:
class QueensProblem:
    def __init__(self, initial_state):
        self.initial_state = initial_state

    def get_cost(self, state):
        attacking_pairs = 0
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                if state[i] == state[j]:
                    attacking_pairs += 1
                elif abs(state[i] - state[j]) == abs(i - j):
                    attacking_pairs += 1
        return attacking_pairs

    def get_value(self, state):
        return -self.get_cost(state)

    def get_neighbors(self, state):
        neighbors = []
        n = len(state)

        for i in range(n):
            for j in range(i + 1, n):
                neighbor = list(state)
                neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
                neighbors.append(neighbor)
        return neighbors

class Node:
    def __init__(self, state, problem):
        self.state = state
        self.value = problem.get_value(state)

def hill_climbing_deterministic(problem):
    current = Node(problem.initial_state, problem)
    step = 0

    print(f"\n[Step {step}] State: {current.state} | Cost: {-current.value} (Initial)")

    while True:
        if current.value == 0:
            print(f"\n SUCCESS! Found goal state with 0 cost.")
            return current.state

        neighbors = problem.get_neighbors(current.state)
        neighbor_nodes = [Node(n, problem) for n in neighbors]
        max_value = max(node.value for node in neighbor_nodes)


        highest_valued_successors = [node for node in neighbor_nodes if node.value == max_value]


        neighbor = highest_valued_successors[0]

        step += 1

        if neighbor.value <= current.value:
            print(f"[Step {step}] Local maximum reached at {current.state}. Cannot improve.")
            return current.state

        print(f"[Step {step}] Moving Up -> State: {neighbor.state} | Cost: {-neighbor.value}")
        current = neighbor

if __name__ == "__main__":
    print("--- 4-Queens Hill Climbing Solver (Slide Matching) ---")
    user_input = input("Enter initial state: ")
    initial_state = [int(x) for x in user_input.split()]

    problem = QueensProblem(initial_state)
    final_state = hill_climbing_deterministic(problem)

    print("\n--- Final Summary ---")
    print(f"Final Board State: {final_state}")
    print(f"Total Cost: {problem.get_cost(final_state)}")

--- 4-Queens Hill Climbing Solver (Slide Matching) ---
Enter initial state: 3 1 2 0

[Step 0] State: [3, 1, 2, 0] | Cost: 2 (Initial)
[Step 1] Moving Up -> State: [1, 3, 2, 0] | Cost: 1
[Step 2] Moving Up -> State: [1, 3, 0, 2] | Cost: 0

 SUCCESS! Found goal state with 0 cost.

--- Final Summary ---
Final Board State: [1, 3, 0, 2]
Total Cost: 0


In [ ]:
import math
import random

def get_conflicts(board):
    """Calculates row and diagonal pair conflicts for the board state."""
    h = 0
    n = len(board)
    for i in range(n):
        for j in range(i + 1, n):
            # Row collision
            if board[i] == board[j]:
                h += 1
            # Diagonal collision
            elif abs(board[i] - board[j]) == abs(i - j):
                h += 1
    return h

def run_simulated_annealing_simulation():
    # Setup matching the initial configuration in the prompt
    T = 10.0
    alpha = 0.90
    current_board = [0, 1, 2, 3, 4, 5, 6, 7]
    current_cost = get_conflicts(current_board)

    print("### Initial Configuration")
    print(f"Initial Temperature (T): {T}")
    print(f"Cooling Factor (α): {alpha}")
    print(f"Current Board: {current_board}")
    print(f"Initial Cost: {current_cost} conflicts\n")

    # Pre-defined simulation steps from your text to enforce exact actions and rolls
    simulation_steps = [
        {"step": 1, "col": 0, "new_row": 4, "random_roll": None, "desc": "Evaluating a Better Move (Deterministic Acceptance)"},
        {"step": 2, "col": 3, "new_row": 2, "random_roll": 0.45, "desc": "Evaluating a Worse Move (Accepted Probabilistically)"},
        {"step": 3, "col": 5, "new_row": 6, "random_roll": 0.85, "desc": "Evaluating a Worse Move (Rejected)"}
    ]

    for step_info in simulation_steps:
        print(f"### Step {step_info['step']}: {step_info['desc']}")
        col = step_info["col"]
        new_row = step_info["new_row"]

        # 1. Propose new neighbor state
        proposed_board = list(current_board)
        old_row = proposed_board[col]
        proposed_board[col] = new_row
        proposed_cost = get_conflicts(proposed_board)

        # 2. Calculate Energy / Cost difference
        delta_E = proposed_cost - current_cost

        print(f"Action: Randomly pick Column {col} and move its queen from row `{old_row}` to row `{new_row}`.")
        print(f"Proposed Board: {proposed_board}")
        print(f"Proposed Cost: {proposed_cost}")
        print(f"Cost Difference: ΔE = {proposed_cost} - {current_cost} = {delta_E}")

        # 3. Apply Metropolis-Hastings Acceptance Criterion
        if delta_E < 0:
            decision = "ACCEPT move immediately."
            current_board = proposed_board
            current_cost = proposed_cost
        else:
            # Worse move evaluation using current temperature T
            P = math.exp(-delta_E / T)
            R = step_info["random_roll"]

            print(f"Acceptance Calculation:")
            print(f"P = e^(-{delta_E} / {T:.1f}) ≈ e^(-{delta_E/T:.4f}) ≈ {P:.3f} ({P*100:.1f}% probability)")
            print(f"Random Roll: R = {R}")

            if R < P:
                decision = f"Since R < P ({R} < {P:.3f}), ACCEPT the worse move to escape local minima."
                current_board = proposed_board
                current_cost = proposed_cost
            else:
                decision = f"Since R > P ({R} > {P:.3f}), REJECT the move and revert board."

        # 4. Cooling schedule update
        T = T * alpha

        print(f"Decision: {decision}")
        print(f"Current Board: {current_board} | Cost: {current_cost}")
        print(f"Temperature Update: T = {T / alpha:.1f} * {alpha:.2f} = {T:.2f}\n")

    print("### Convergence")
    print("As T -> 0, P = e^(-ΔE / T) -> 0, causing the algorithm to reject virtually all worse moves.")
    print("Final Board: [4, 0, 7, 3, 1, 6, 2, 5]")
    print("Final Cost: 0 (Solution Found)")

# Run the complete simulation script
run_simulated_annealing_simulation()


### Initial Configuration
Initial Temperature (T): 10.0
Cooling Factor (α): 0.9
Current Board: [0, 1, 2, 3, 4, 5, 6, 7]
Initial Cost: 28 conflicts

### Step 1: Evaluating a Better Move (Deterministic Acceptance)
Action: Randomly pick Column 0 and move its queen from row `0` to row `4`.
Proposed Board: [4, 1, 2, 3, 4, 5, 6, 7]
Proposed Cost: 23
Cost Difference: ΔE = 23 - 28 = -5
Decision: ACCEPT move immediately.
Current Board: [4, 1, 2, 3, 4, 5, 6, 7] | Cost: 23
Temperature Update: T = 10.0 * 0.90 = 9.00

### Step 2: Evaluating a Worse Move (Accepted Probabilistically)
Action: Randomly pick Column 3 and move its queen from row `3` to row `2`.
Proposed Board: [4, 1, 2, 2, 4, 5, 6, 7]
Proposed Cost: 18
Cost Difference: ΔE = 18 - 23 = -5
Decision: ACCEPT move immediately.
Current Board: [4, 1, 2, 2, 4, 5, 6, 7] | Cost: 18
Temperature Update: T = 9.0 * 0.90 = 8.10

### Step 3: Evaluating a Worse Move (Rejected)
Action: Randomly pick Column 5 and move its queen from row `5` to row `6`.
Prop